### Classification 
{SUPPORTS, REFUTES, NOT_ENOUGH_INFO, DISPUTED}.


-[train-claims,dev-claims].json: JSON files for the labelled training and development set;

-[test-claims-unlabelled].json: JSON file for the unlabelled test set;

-evidence.json: JSON file containing a large number of evidence passages (i.e. the “knowledge source”);

-dev-claims-baseline.json: JSON file containing predictions of a baseline system on the development set;
-eval.py: Python script to evaluate system performance (see “Evaluation” below for more details).

In [29]:
import json
import numpy as np
import re

import time

import time
from datetime import datetime
import os
import pandas as pd


from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [3]:
dev_baseline = 'data\dev-claims-baseline.json'
dev_claims = 'data\dev-claims.json'

train_claims_path = 'data/train-claims.json'
test_claims_path = 'data/test-claims-unlabelled.json'

evidence_path = 'data\evidence.json'


### Read and storage Data

In [4]:
# For labelled data
def load_json_to_dataframe(json_file):
    # Load the JSON file
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Create empty lists to store the data
    ids = []
    claim_texts = []
    claim_labels = []
    evidences = []

    # Iterate over each claim in the JSON data
    for claim_id, claim_data in data.items():
        # Extract claim details
        ids.append(claim_id.split('-')[1])
        claim_texts.append(claim_data["claim_text"])
        claim_labels.append(claim_data["claim_label"])
        evidences.append(claim_data["evidences"])

    # Create a DataFrame
    df = pd.DataFrame({
        'id': ids,
        'claim_text': claim_texts,
        'claim_label': claim_labels,
        'evidences': evidences
    })

    return df


def load_evidnece(json_file):
    # 读取 JSON 文件
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 创建一个空的 DataFrame
    df = pd.DataFrame(data.items(), columns=['id', 'text'])
    
    return df

def load_test(json_file):
    # Load the JSON file
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Create empty lists to store the data
    ids = []
    claim_texts = []
    
    # Iterate over each claim in the JSON data
    for claim_id, claim_data in data.items():
        # Extract claim details
        ids.append(claim_id.split('-')[1])
        claim_texts.append(claim_data["claim_text"])
        

    # Create a DataFrame
    df = pd.DataFrame({
        'id': ids,
        'claim_text': claim_texts,
    })

    return df



In [13]:
# 检查有没有少读
def count_claim_entries(json_file):
    # 读取 JSON 文件
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 初始化计数器
    count = 0

    # 遍历 JSON 数据，计算具有指定 "claim_id" 的条目数
    for key in data:
        
        count += 1

    return count

# 调用函数并打印结果
count = count_claim_entries(test_claims_path)
print(f"The number of entries is: {count}")


The number of entries is: 153


In [12]:
df_test = load_test(test_claims_path)
df_test 

,id,claim_text
0,2967,The contribution of waste heat to the global c...
1,979,“Warm weather worsened the most recent five-ye...
2,1609,Greenland has only lost a tiny fraction of its...
3,1020,“The global reef crisis does not necessarily m...
4,2599,Small amounts of very active substances can ca...
...,...,...
148,293,When the measuring equipment gets old and need...
149,910,"The cement, iron and steel, and petroleum refi..."
150,2815,A new peer-reviewed study on Surface Warming a...
151,1652,The strong CO2 effect has been observed by man...


In [7]:
df_evidence = load_evidnece(evidence_path)
row1 = df_evidence.loc[df_evidence['id'] == 'evidence-67732']
row1.iloc[0, 1]


'[citation needed] South Australia has the highest retail price for electricity in the country.'

In [8]:
row2 = df_evidence.loc[df_evidence['id'] == 'evidence-572512']
row2.iloc[0, 1]

'"South Australia has the highest power prices in the world".'

In [20]:
df_dv = load_json_to_dataframe(dev_claims)
df_dv.head(1)

,id,claim_text,claim_label,evidences
0,752,[South Australia] has the most expensive elect...,SUPPORTS,"[evidence-67732, evidence-572512]"


In [10]:
df_dv.head(1)

,id,claim_text,claim_label,evidences
0,752,[South Australia] has the most expensive elect...,SUPPORTS,"[evidence-67732, evidence-572512]"


In [11]:
df_train = load_json_to_dataframe('data/train-claims.json')
df_train.head(5)
    

,id,claim_text,claim_label,evidences
0,1937,Not only is there no scientific evidence that ...,DISPUTED,"[evidence-442946, evidence-1194317, evidence-1..."
1,126,El Niño drove record highs in global temperatu...,REFUTES,"[evidence-338219, evidence-1127398]"
2,2510,"In 1946, PDO switched to a cool phase.",SUPPORTS,"[evidence-530063, evidence-984887]"
3,2021,Weather Channel co-founder John Coleman provid...,DISPUTED,"[evidence-1177431, evidence-782448, evidence-5..."
4,2449,"""January 2008 capped a 12 month period of glob...",NOT_ENOUGH_INFO,"[evidence-1010750, evidence-91661, evidence-72..."


In [22]:
df_dv['evidence_texts'] = df_dv['evidences'].apply(
    lambda ids: [df_evidence[df_evidence['id'] == evidence]['text'].iloc[0] for evidence in ids]
)


In [25]:
df_dv.head(1)

,id,claim_text,claim_label,evidences,evidence_texts
0,752,[South Australia] has the most expensive elect...,SUPPORTS,"[evidence-67732, evidence-572512]",[[citation needed] South Australia has the hig...


In [28]:
df_dv.iloc[0, 4]

['[citation needed] South Australia has the highest retail price for electricity in the country.',
 '"South Australia has the highest power prices in the world".']

### Build models

In [41]:

from sklearn.feature_extraction.text import CountVectorizer



# 假设df是你的DataFrame，包含claim_text, claim_label和evidence_texts列

# 将claim_text和evidence_texts列合并为一列，作为模型的输入文本
X = df_dv['claim_text'] + df_dv['evidence_texts'].apply(lambda x: ' '.join(x))
y = df_dv['claim_label']

# 划分数据集为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 使用CountVectorizer向量化文本
count_vectorizer = CountVectorizer()

# 在训练集上拟合CountVectorizer，并转换训练集和测试集
X_train_count = count_vectorizer.fit_transform(X_train)
X_test_count = count_vectorizer.transform(X_test)

# 定义要尝试的C值
C_values = [0.01, 0.1, 1.0, 10.0]

# 创建空列表来存储每个C值的准确率
accuracy_scores = []

# 在不同的C值下训练模型并评估性能
for C in C_values:
    # 训练逻辑回归分类器
    classifier = LogisticRegression(max_iter=1000, C=C)
    classifier.fit(X_train_count, y_train)
    
    # 预测测试集
    y_pred = classifier.predict(X_test_count)
    
    # 计算准确率并添加到列表中
    accuracy = accuracy_score(y_test, y_pred)
    accuracy_scores.append(accuracy)
    
    # 打印当前C值的准确率
    print(f"C = {C}: Accuracy = {accuracy}")

# 打印不同C值下的准确率
print("Accuracy scores:", accuracy_scores)


C = 1.0: Accuracy = 0.4838709677419355
Accuracy scores: [0.4838709677419355]


In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 定义要尝试的参数组合
n_estimators_values = [50, 100, 200]
max_depth_values = [None, 10, 20]

# 创建空列表来存储每个参数组合的准确率
accuracy_scores_rf = []

# 在不同的参数组合下训练模型并评估性能
for n_estimators in n_estimators_values:
    for max_depth in max_depth_values:
        # 创建随机森林分类器
        rf_classifier = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        
        # 在训练集上拟合模型
        rf_classifier.fit(X_train_count, y_train)
        
        # 在测试集上进行预测
        y_pred_rf = rf_classifier.predict(X_test_count)
        
        # 计算准确率并添加到列表中
        accuracy_rf = accuracy_score(y_test, y_pred_rf)
        accuracy_scores_rf.append(((n_estimators, max_depth), accuracy_rf))
        
        # 打印当前参数组合的准确率
        print(f"n_estimators = {n_estimators}, max_depth = {max_depth}: Accuracy = {accuracy_rf}")

# 打印不同参数组合下的准确率
print("Accuracy scores for Random Forest:")
for params, accuracy in accuracy_scores_rf:
    print(f"Parameters: {params}, Accuracy: {accuracy}")


n_estimators = 50, max_depth = None: Accuracy = 0.2903225806451613
n_estimators = 50, max_depth = 10: Accuracy = 0.3225806451612903
n_estimators = 50, max_depth = 20: Accuracy = 0.2903225806451613
n_estimators = 100, max_depth = None: Accuracy = 0.3225806451612903
n_estimators = 100, max_depth = 10: Accuracy = 0.3225806451612903
n_estimators = 100, max_depth = 20: Accuracy = 0.3225806451612903
n_estimators = 200, max_depth = None: Accuracy = 0.3225806451612903
n_estimators = 200, max_depth = 10: Accuracy = 0.3225806451612903
n_estimators = 200, max_depth = 20: Accuracy = 0.3548387096774194
Accuracy scores for Random Forest:
Parameters: (50, None), Accuracy: 0.2903225806451613
Parameters: (50, 10), Accuracy: 0.3225806451612903
Parameters: (50, 20), Accuracy: 0.2903225806451613
Parameters: (100, None), Accuracy: 0.3225806451612903
Parameters: (100, 10), Accuracy: 0.3225806451612903
Parameters: (100, 20), Accuracy: 0.3225806451612903
Parameters: (200, None), Accuracy: 0.3225806451612903
P

In [33]:
from sklearn.svm import SVC


# 定义要尝试的参数组合
C_values = [0.01, 0.1, 1.0, 10.0]
kernel_values = ['linear', 'rbf']

# 创建空列表来存储每个参数组合的准确率
accuracy_scores_svm = []

# 在不同的参数组合下训练模型并评估性能
for C in C_values:
    for kernel in kernel_values:
        # 创建SVM分类器
        svm_classifier = SVC(C=C, kernel=kernel, random_state=42)
        
        # 在训练集上拟合模型
        svm_classifier.fit(X_train_count, y_train)
        
        # 在测试集上进行预测
        y_pred_svm = svm_classifier.predict(X_test_count)
        
        # 计算准确率并添加到列表中
        accuracy_svm = accuracy_score(y_test, y_pred_svm)
        accuracy_scores_svm.append(((C, kernel), accuracy_svm))
        
        # 打印当前参数组合的准确率
        print(f"C = {C}, kernel = {kernel}: Accuracy = {accuracy_svm}")

# 打印不同参数组合下的准确率
print("Accuracy scores for SVM:")
for params, accuracy in accuracy_scores_svm:
    print(f"Parameters: {params}, Accuracy: {accuracy}")


C = 0.01, kernel = linear: Accuracy = 0.45161290322580644
C = 0.01, kernel = rbf: Accuracy = 0.2903225806451613
C = 0.1, kernel = linear: Accuracy = 0.45161290322580644
C = 0.1, kernel = rbf: Accuracy = 0.2903225806451613
C = 1.0, kernel = linear: Accuracy = 0.45161290322580644
C = 1.0, kernel = rbf: Accuracy = 0.45161290322580644
C = 10.0, kernel = linear: Accuracy = 0.45161290322580644
C = 10.0, kernel = rbf: Accuracy = 0.4838709677419355
Accuracy scores for SVM:
Parameters: (0.01, 'linear'), Accuracy: 0.45161290322580644
Parameters: (0.01, 'rbf'), Accuracy: 0.2903225806451613
Parameters: (0.1, 'linear'), Accuracy: 0.45161290322580644
Parameters: (0.1, 'rbf'), Accuracy: 0.2903225806451613
Parameters: (1.0, 'linear'), Accuracy: 0.45161290322580644
Parameters: (1.0, 'rbf'), Accuracy: 0.45161290322580644
Parameters: (10.0, 'linear'), Accuracy: 0.45161290322580644
Parameters: (10.0, 'rbf'), Accuracy: 0.4838709677419355


In [34]:
df_train['evidence_texts'] = df_train['evidences'].apply(
    lambda ids: [df_evidence[df_evidence['id'] == evidence]['text'].iloc[0] for evidence in ids]
)

X_training = df_train['claim_text'] + df_train['evidence_texts'].apply(lambda x: ' '.join(x))
y_training = df_train['claim_label']



In [47]:
# X_train_count2 = count_vectorizer.fit_transform(X_training)
X_train_count2 = count_vectorizer.transform(X_training)



### Apply on training set

In [48]:

# 训练逻辑回归分类器
classifier2 = LogisticRegression(max_iter=1000, C=0.1)
classifier2.fit(X_train_count, y_train)
    
# 预测测试集
y_pred2 = classifier2.predict(X_train_count2)
    
    # 计算准确率并添加到列表中
accuracy = accuracy_score(y_training , y_pred2)

# 打印不同C值下的准确率
print("Accuracy scores:", accuracy)






Accuracy scores: 0.5252442996742671
